# MCDI500 · Fase 2 — Preprocesamiento, validación y métricas de concentración
**Proyecto:** Distribución y concentración de matrícula de primer año de Institutos Profesionales en Chile durante 2025. **Fuente:** CNED, misma utilizada en F1.

**Pregunta (idéntica a F1 e informe):** ¿Cómo se distribuye y concentra la matrícula de primer año de los Institutos Profesionales en Chile durante 2025, según institución, región y área de conocimiento?

**Objetivo general (idéntico a F1 e informe):** Analizar la distribución y la concentración de la matrícula de primer año de los Institutos Profesionales en Chile durante 2025, por institución, región y área de conocimiento, mediante un flujo de procesamiento y análisis descriptivo reproducible en Python.

**Proceso:** cargar extracto F1 → perfil de calidad → descartar columnas completamente vacías y >90% ausentes → comparar tratamientos → generar variables derivadas sin alterar observaciones → validar → resumir por institución, región y área → exportar evidencias. Se adopta el esquema técnico del repositorio docente, adaptado al significado de las variables CNED.

## 1. Entorno, rutas, dependencias y funciones compartidas
F2 obtiene los registros desde el Excel **sin depender de productos generados en F1**. Usa `src/cned_pipeline.py`, fija la semilla y registra versiones para una ejecución reproducible. No instala paquetes dentro del notebook; la instalación corresponde a `requirements.txt`.


In [1]:
from pathlib import Path
import json,sys,subprocess
from importlib import metadata as imeta
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
np.random.seed(42)
RAIZ=Path.cwd().resolve()
if RAIZ.name in {"F1","F2","F3","F4"}: RAIZ=RAIZ.parent
if not (RAIZ/"src/cned_pipeline.py").exists():raise FileNotFoundError("No existe el módulo compartido: src/cned_pipeline.py")
if str(RAIZ) not in sys.path:sys.path.insert(0,str(RAIZ))
from src.cned_pipeline import (cargar_cned_ip, diagnosticar_faltantes,
    depurar_estructura, comparar_imputaciones, preparar_fase2, validar_fase1,
    validar_fase2, concentracion_matricula, promedio_matricula, diagnosticar_atipicos)
RUTA_EXCEL=RAIZ/'data/raw/Base-INDICES-Matricula-2005-2026.xlsx'
DIR_INTERIM=RAIZ/'data/interim';DIR_PROCESADO=RAIZ/'data/processed'
DIR_DOCS=RAIZ/'docs'; DIR_EVIDENCIAS=RAIZ/'evidencias'
for carpeta in (DIR_INTERIM,DIR_PROCESADO,DIR_DOCS,DIR_EVIDENCIAS):carpeta.mkdir(parents=True,exist_ok=True)
assert RUTA_EXCEL.is_file(),f"Falta el Excel: {RUTA_EXCEL}"
print("Intérprete:",sys.executable,"| Python:",sys.version.split()[0],"| virtualenv:",sys.prefix!=sys.base_prefix)
for p in ["pandas","numpy","openpyxl","lxml","matplotlib","jupyterlab","ipykernel","nbformat","nbclient","pytest"]:
    print(f"[OK] {p:12} {imeta.version(p)}")


Intérprete: /opt/pyvenv/bin/python | Python: 3.13.5 | virtualenv: True
[OK] pandas       2.2.3
[OK] numpy        2.3.5
[OK] openpyxl     3.1.5
[OK] lxml         6.1.1
[OK] matplotlib   3.10.8
[OK] jupyterlab   4.5.3
[OK] ipykernel    7.2.0
[OK] nbformat     5.10.4
[OK] nbclient     0.10.4
[OK] pytest       9.0.2


## 2. Obtención y comprobación del dataset IP/2025
A partir del libro original se selecciona el subconjunto con `Año=2025` y `Tipo Institución=I.P.`. F2 guarda una copia de trabajo en `data/interim/` sin modificar la fuente; registra su cantidad de registros, variables y matrícula. Esta etapa es **obtención**, previa a limpieza.


In [2]:
original,proveniencia=cargar_cned_ip(RUTA_EXCEL)
RUTA_EXTRACTO=DIR_INTERIM/'ip_2025_origen_filtrado.csv'
original.to_csv(RUTA_EXTRACTO,index=False,encoding='utf-8-sig')
releido_origen=pd.read_csv(RUTA_EXTRACTO,encoding='utf-8-sig',low_memory=False)
assert releido_origen.shape==original.shape==(4731,62)
base=validar_fase1(original)
print(json.dumps(base,ensure_ascii=False,indent=2))
display(original[["Nombre Institución","Nombre Region","Area Conocimiento","Matrícula Primer Año","Matrícula Total"]].head())
assert base["matricula_primer_anio"]==137481 and base["matricula_total"]==439607
print("Exportado el subconjunto sin modificaciones:",RUTA_EXTRACTO)


{
  "registros": 4731,
  "instituciones": 27,
  "columnas": 62,
  "matricula_primer_anio": 137481,
  "matricula_total": 439607,
  "registro_inicio_posterior": 1,
  "duplicados_completos": 0,
  "matricula_primer_anio_cero": 1576,
  "pregrado": 4714,
  "posgrado": 17
}


,Nombre Institución,Nombre Region,Area Conocimiento,Matrícula Primer Año,Matrícula Total
0,IP AGRARIO ADOLFO MATTHEI,Región de Los Lagos,"Agricultura, Silvicultura, Pesca y Veterinaria",72,367
1,IP AGRARIO ADOLFO MATTHEI,Región de Los Lagos,"Agricultura, Silvicultura, Pesca y Veterinaria",41,144
2,IP AGRARIO ADOLFO MATTHEI,Región de Los Lagos,"Agricultura, Silvicultura, Pesca y Veterinaria",69,166
3,IP AGRARIO ADOLFO MATTHEI,Región de Los Lagos,"Agricultura, Silvicultura, Pesca y Veterinaria",41,79
4,IP INACAP,Región de Antofagasta,Administración y Comercio,29,165


Exportado el subconjunto sin modificaciones: /mnt/data/CNED_MCDI500_Sumativa1_Guia_Integrada/data/interim/ip_2025_origen_filtrado.csv


## 3. Diagnóstico previo: dimensiones, tipos, faltantes, duplicados y extremos
Se cuantifican los nulos **antes** de cualquier limpieza. Las columnas vacías o con ausencia superior a 90% se estudian mediante una regla explícita de F2, sin perder variables críticas de matrícula. Los valores extremos se identifican con IQR sobre matrículas, vacantes y aranceles CLP, **pero no se eliminan automáticamente**: pueden representar ofertas genuinamente grandes.


In [3]:
perfil_antes=diagnosticar_faltantes(original)
display(perfil_antes.head(18))
limpio,aud_estructura=depurar_estructura(original)
print("Completamente vacías:",len(aud_estructura["columnas_vacias"]),aud_estructura["columnas_vacias"])
print(">90% de faltantes:",aud_estructura["columnas_excluidas_por_umbral"])
print("Columnas retenidas:",limpio.shape[1])
print("Duplicados exactos:",int(original.duplicated().sum()))
tabla_atipicos=diagnosticar_atipicos(limpio)
display(tabla_atipicos)
assert len(aud_estructura["columnas_vacias"])==13
assert set(aud_estructura["columnas_excluidas_por_umbral"])=={"Grado Académico","Mención o Especialidad"}
perfil_antes.to_csv(DIR_DOCS/"faltantes_antes.csv",index=False,encoding='utf-8-sig')
tabla_atipicos.to_csv(DIR_DOCS/"diagnostico_atipicos.csv",index=False,encoding='utf-8-sig')


,variable,faltantes,porcentaje_faltante,tipo
0,Máximo Puntaje (promedio matemáticas y lenguaje),4731,100.000,string
1,Máximo Puntaje NEM,4731,100.000,string
2,Máximo Puntaje Ranking,4731,100.000,string
3,Mínimo Puntaje (promedio matemáticas y lenguaje),4731,100.000,string
4,Mínimo Puntaje NEM,4731,100.000,string
5,Mínimo Puntaje Ranking,4731,100.000,string
6,Nº Alumnos Ingreso Via PSU o PDT,4731,100.000,float64
7,Promedio Puntaje (promedio matemáticas y lengu...,4731,100.000,string
8,Promedio Puntaje NEM,4731,100.000,string
9,Promedio Puntaje Ranking,4731,100.000,string


Completamente vacías: 13 ['Máximo Puntaje (promedio matemáticas y lenguaje)', 'Promedio Puntaje (promedio matemáticas y lenguaje)', 'Mínimo Puntaje (promedio matemáticas y lenguaje)', 'Puntaje de corte (primer seleccionado)', 'Puntaje de corte (promedio de la carrera)', 'Puntaje de corte (último seleccionado)', 'Máximo Puntaje NEM', 'Promedio Puntaje NEM', 'Mínimo Puntaje NEM', 'Máximo Puntaje Ranking', 'Promedio Puntaje Ranking', 'Mínimo Puntaje Ranking', 'Nº Alumnos Ingreso Via PSU o PDT']
>90% de faltantes: ['Mención o Especialidad', 'Grado Académico']
Columnas retenidas: 48
Duplicados exactos: 0


,variable,n_observados,q1,q3,iqr,limite_inferior,limite_superior,atipicos,pct_atipicos
0,Matrícula Primer Año,4731,0.0,34.0,34.0,-51.0,85.0,304,6.426
1,Matrícula Total,4731,14.0,101.0,87.0,-116.5,231.5,402,8.497
2,Vacantes,3835,20.0,40.0,20.0,-10.0,70.0,459,11.969
3,Valor de arancel,3150,2230000.0,2950000.0,720000.0,1150000.0,4030000.0,37,1.175


## 4. Comparación y elección de imputaciones
**Se comparan cuatro alternativas:** eliminar filas faltantes (se informa cuántas se perderían), media global, mediana global y mediana jerárquica por institución/área. Las tres alternativas de imputación se evalúan ocultando 10% de valores **observados** (semilla 42) y calculando el MAE. La decisión se toma por variable según el error y el significado del dato, no por una receta única.

**Reglas de dominio:** las variables monetarias en UF no se completan con pesos. Códigos, matrícula de primer año y matrícula total no se imputan. Toda estimación se guarda en una columna `*_analisis` con bandera `*_imputado`, preservando el original. Se mide además el efecto de imputar sobre la desviación estándar y se documenta el sesgo potencial de no aleatoriedad de ausencias (Oberman y Vink, 2024).


In [4]:
comparaciones=[comparar_imputaciones(limpio,campo,semilla=42)
   for campo in ("Valor de arancel","Valor de matrícula","Valor del Título","Vacantes")]
comparacion_df=pd.DataFrame(comparaciones)
display(comparacion_df)
assert comparacion_df["metodo_seleccionado"].isin(["media","global","jerarquica"]).all()
assert (comparacion_df["filas_perdidas_si_elimina"]>0).any()


,variable,observaciones_validacion,mae_media,mae_global,mae_jerarquica,metodo_seleccionado,filas_perdidas_si_elimina,semilla
0,Valor de arancel,315,421312.015,419175.098,204107.098,jerarquica,1571,42
1,Valor de matrícula,330,72548.286,66149.697,31583.030,jerarquica,1417,42
2,Valor del Título,424,272892.993,248201.866,64938.297,jerarquica,475,42
3,Vacantes,383,41.087,33.478,25.764,jerarquica,896,42


### 4.1 Aplicación justificada y medición del efecto
El pipeline conserva los valores observados, genera estimaciones por variable y mide el cambio en desviación estándar antes/después. Si la media o mediana global reduce excesivamente la variabilidad, la comparación con mediana jerárquica permite justificar la alternativa. La validación se ejecuta inmediatamente.

In [5]:
final, aud=preparar_fase2(original)
resultado=validar_fase2(original,final,aud)
print(json.dumps(resultado,ensure_ascii=False,indent=2))
display(aud['estrategias'])
assert resultado['filas_sin_perdidas']==4731
assert final['Matrícula Primer Año'].sum()==137481
assert final['Matrícula Total'].sum()==439607

{
  "filas_sin_perdidas": 4731,
  "columnas_salida": 68,
  "filas_anomalia_temporal": 1,
  "cero_primer_anio_preservados": 1576,
  "nulos_originales_preservados_por_columna": true,
  "desgloses_matricula_consistentes": true
}


,variable,observaciones_validacion,mae_media,mae_global,mae_jerarquica,metodo_seleccionado,filas_perdidas_si_elimina,semilla,faltantes_originales,estimaciones_realizadas,faltantes_final_analisis,desv_antes_observados,desv_despues_analisis,cambio_desv_pct
0,Valor de arancel,315,421312.015,419175.098,204107.098,jerarquica,1571,42,1576,1571,5,519641.174,481176.638,-7.402
1,Valor de matrícula,330,72548.286,66149.697,31583.030,jerarquica,1417,42,1422,1417,5,97367.092,89915.742,-7.653
2,Valor del Título,424,272892.993,248201.866,64938.297,jerarquica,475,42,475,475,0,308861.110,303194.999,-1.835
3,Vacantes,383,41.087,33.478,25.764,jerarquica,896,42,896,896,0,98.020,90.268,-7.909


## 5. Transformaciones pertinentes a cada rol analítico
1. **Casting y normalización textual:** marcadores `Null` a ausente, `Tipo Moneda` a CLP/UF.
2. **Continuas:** los conteos reales permanecen intactos; se agrega `log1p` y la escala derivada 0–1 para control visual, sin reemplazar los resultados.
3. **Nominales:** `Pregrado/Posgrado` se transforma en dos indicadores *one-hot*, y se verifica que su suma sea 1 por fila.
4. **Ordinales:** los cuartiles empíricos de `Matrícula Total` tienen el orden **declarado** Baja → Media → Alta → Muy alta; su código es explícito y no deriva de orden alfabético.
5. **Trazabilidad:** banderas de imputación y de inicio posterior a 2025, manteniendo el registro de Santo Tomás para sensibilidad.

Estas transformaciones auxiliares **no afectan** las sumas de matrícula ni sustituyen observaciones originales. Se comparan métodos por su pertinencia y resultado cuantificado, como exige la rúbrica.


In [6]:
tabla_categorias=final['categoria_matricula_total'].value_counts(dropna=False).sort_index().rename_axis('categoria').reset_index(name='registros')
display(tabla_categorias)
print("Registros UF:",int(final['moneda_normalizada'].eq('UF').sum()))
print("Ceros primer año:",int(final['matricula_primer_anio_cero'].sum()))
print("Anomalías temporales:",int(final['inicio_posterior_a_2025'].sum()))
print("Nulos restantes (legítimos según variable):",int(final.isna().sum().sum()))
display(final.isna().sum().sort_values(ascending=False).head(12).to_frame('faltantes'))
assert (final[['nivel_pregrado_onehot','nivel_posgrado_onehot']].sum(axis=1)==1).all()
assert final['categoria_matricula_total_codigo'].between(1,4).all()
assert final['matricula_primer_anio_escala_0_1'].between(0,1).all()
assert np.allclose(final['matricula_primer_anio_log1p'], np.log1p(final['Matrícula Primer Año']))


,categoria,registros
0,Baja,1223
1,Media,1169
2,Alta,1165
3,Muy alta,1174


Registros UF: 10
Ceros primer año: 1576
Anomalías temporales: 1
Nulos restantes (legítimos según variable): 12100


,faltantes
Nombre del Campus,2747
Cód. Campus,2674
Nº Alumnos Ingreso Otra Via,1593
Valor de arancel,1576
Valor de matrícula,1422
Vacantes,896
Códgo SIES,705
Valor del Título,475
valor_matricula_analisis,5
arancel_analisis,5


## 6. Indicadores descriptivos de la pregunta de investigación
Se calculan participaciones y concentración **sobre matrícula de primer año observada**, no sobre el número de filas. Se reporta CR3 (porcentaje de la matrícula concentrada en los tres grupos mayores) e índice de Herfindahl-Hirschman (HHI, escala 0–10.000). HHI describe concentración, no demuestra causalidad ni calidad educativa. Para región y área se calculan además tablas completas, no solo máximos.

In [7]:
dimensiones={
    "institucion":"Nombre Institución", 
    "region":"Nombre Region", 
    "area":"Area Conocimiento"
}
indicadores=[]
for clave,campo in dimensiones.items():
    tabla, resumen=concentracion_matricula(final,campo)
    indicadores.append(resumen)
    tabla.to_csv(DIR_PROCESADO/f'matricula_por_{clave}.csv',index=False,encoding='utf-8-sig')
    print("\n",campo,"| CR3:",resumen['cr3_pct'],"| HHI:",resumen['hhi'])
    display(tabla.head(8))
resumen_concentracion=pd.DataFrame(indicadores)
display(resumen_concentracion)


 Nombre Institución | CR3: 53.916 | HHI: 1348.247


,Nombre Institución,matricula_primer_anio,registros,participacion_pct
0,IP DUOC UC,33104,854,24.0790
1,IP AIEP,25422,1635,18.4913
2,IP LATINOAMERICANO DE COMERCIO EXTERIOR,15598,125,11.3456
3,IP INACAP,14088,739,10.2472
4,IP SANTO TOMÁS,11244,385,8.1786
5,IP IACC,11032,53,8.0244
6,IP PROVIDENCIA,6501,46,4.7287
7,IP DE CHILE,5946,209,4.3250



 Nombre Region | CR3: 79.243 | HHI: 4402.463


,Nombre Region,matricula_primer_anio,registros,participacion_pct
0,Región Metropolitana,89674,2106,65.2265
1,Región del Bío-Bío,10085,574,7.3356
2,Región de Valparaíso,9185,432,6.6809
3,Región de Antofagasta,4180,203,3.0404
4,Región de Los Lagos,4136,260,3.0084
5,Región del Gral. B. O`Higgins,4081,251,2.9684
6,Región de La Araucanía,3739,171,2.7196
7,Región de Coquimbo,3652,179,2.6564



 Area Conocimiento | CR3: 79.806 | HHI: 2862.56


,Area Conocimiento,matricula_primer_anio,registros,participacion_pct
0,Tecnología,61423,1958,44.6774
1,Administración y Comercio,35461,1254,25.7934
2,Salud,12834,521,9.3351
3,Educación,11054,382,8.0404
4,Ciencias Sociales,7527,262,5.4749
5,Arte y Arquitectura,5614,226,4.0835
6,"Agricultura, Silvicultura, Pesca y Veterinaria",2042,58,1.4853
7,Derecho,907,38,0.6597


,dimension,total_matricula_primer_anio,grupos,cr3_pct,hhi
0,Nombre Institución,137481,27,53.916,1348.247
1,Nombre Region,137481,16,79.243,4402.463
2,Area Conocimiento,137481,10,79.806,2862.560


### 6.1 Visualización descriptiva de control
Se presenta el Top 10 de instituciones mediante la matrícula de primer año **observada**, no imputada. La figura no es una predicción y no modifica el dataset; se guarda como evidencia de la Fase 2 y como insumo de F4.

In [8]:
tabla_institucion=pd.read_csv(DIR_PROCESADO/'matricula_por_institucion.csv')
fig,ax=plt.subplots(figsize=(9,5))
top=tabla_institucion.head(10).iloc[::-1]
ax.barh(top['Nombre Institución'],top['matricula_primer_anio'])
ax.set(title='Matrícula de primer año por institución — 2025 (Top 10)',xlabel='Matrícula de primer año registrada')
plt.tight_layout()
fig.savefig(DIR_EVIDENCIAS/'top10_matricula_instituciones.png',dpi=160,bbox_inches='tight')
plt.show()

/tmp/ipykernel_5580/1120515690.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Validaciones normales, límite y excepciones
Las pruebas unitarias en `tests/` verifican: clasificación de cuartiles con empates, errores al promediar datos vacíos, matriz con toda la matrícula ausente, ausencia de valores negativos, UF/CLP, imputaciones y concentración. El notebook confirma de nuevo que **no desaparecieron filas**, las cifras de matrícula no cambiaron y todos los valores observados de aranceles/vacantes permanecen intactos.

**Análisis de sensibilidad:** exclusión del único registro con inicio posterior a 2025 para medir su impacto sobre la matrícula de primer año, sin eliminarlo de la base final.

In [9]:
anom=final.loc[final['inicio_posterior_a_2025']]
base_total=int(final['Matrícula Primer Año'].sum())
sin_anom=int(final.loc[~final['inicio_posterior_a_2025'],'Matrícula Primer Año'].sum())
impacto=base_total-sin_anom
print("Total base:",base_total)
print("Total sin registro temporal atípico:",sin_anom)
print("Diferencia:",impacto,"(%.4f%% del total)" %(100*impacto/base_total))
assert base_total==137481
assert len(anom)==1
# Pruebas rápidas de excepciones de la función compartida.
try:
    promedio_matricula(pd.DataFrame({'Matrícula Total':[None,None]}))
    raise AssertionError('No lanzó excepción cuando toda la matrícula era NA')
except ValueError as error:
    print('[OK] Caso límite NA:',error)
try:
    concentracion_matricula(pd.DataFrame({'Matrícula Primer Año':[-1],'Nombre Institución':['X']}))
    raise AssertionError('No rechazó matrícula negativa')
except ValueError as error:
    print('[OK] Excepción valores negativos:',error)

Total base: 137481
Total sin registro temporal atípico: 137442
Diferencia: 39 (0.0284% del total)
[OK] Caso límite NA: Todos los registros de matrícula están ausentes
[OK] Excepción valores negativos: La matrícula de primer año no es íntegra o contiene negativos


## 8. Persistencia, metadatos y auditoría final
Exportación única del CSV final y tablas de auditoría, guardadas fuera de `data/raw`. Se relee el CSV para comprobar dimensiones y totales, dejando trazabilidad en `docs/`. Los resultados derivados pueden regenerarse mediante F1 → F2.

In [10]:
archivo_final=DIR_PROCESADO/'matricula_ip_2025_procesada_v3.csv'
final.to_csv(archivo_final,index=False,encoding='utf-8-sig')
releido=pd.read_csv(archivo_final,encoding='utf-8-sig',low_memory=False)
assert releido.shape==final.shape
assert int(releido['Matrícula Primer Año'].sum())==137481
assert int(releido['Matrícula Total'].sum())==439607
aud['estrategias'].to_csv(DIR_DOCS/'comparacion_imputaciones.csv',index=False,encoding='utf-8-sig')
tabla_categorias.to_csv(DIR_DOCS/'categorias_matricula.csv',index=False,encoding='utf-8-sig')
resumen_concentracion.to_csv(DIR_DOCS/'indicadores_concentracion.csv',index=False,encoding='utf-8-sig')
reporte={
    'fase':'F2','registros':len(final),'columnas_finales':final.shape[1],
    'matricula_primer_anio':137481,'matricula_total':439607,
    'columnas_vacias':aud['columnas_vacias'],
    'columnas_excluidas_por_umbral':aud['columnas_excluidas_por_umbral'],
    'evaluacion_imputaciones':aud['estrategias'].to_dict('records'),
    'escalamiento':aud['escalamiento'],
    'validacion':resultado,'impacto_anomalia_primer_anio':impacto,
    'indicadores_concentracion':indicadores,
    'nota':'Matrícula original no imputada; datos derivados identificados.'
}
(DIR_DOCS/'metadatos_fase2.json').write_text(json.dumps(reporte,ensure_ascii=False,indent=2),encoding='utf-8')
print('CSV final validado:',archivo_final,'|',final.shape)
print('Resúmenes y evidencias:',DIR_DOCS)


CSV final validado: /mnt/data/CNED_MCDI500_Sumativa1_Guia_Integrada/data/processed/matricula_ip_2025_procesada_v3.csv | (4731, 68)
Resúmenes y evidencias: /mnt/data/CNED_MCDI500_Sumativa1_Guia_Integrada/docs


## 9. Vinculación con el mapa conceptual y continuidad F3–F4
| Tipo requerido por la guía | Evidencia | Estado |
|---|---|---|
| Herramientas científicas | Librerías verificadas, `src/cned_pipeline.py` | Implementado |
| Procesos de calidad | `docs/faltantes_antes.csv`, `docs/diagnostico_atipicos.csv` | Implementado |
| Flujo reproducible | Fuente original → F2 → `data/interim` → `data/processed` | Implementado |
| Decisiones técnicas | `docs/comparacion_imputaciones.csv`, `docs/Registro_decisiones.md` | Implementado |
| Organización del repositorio | `README.md`, carpetas `F1/`, `F2/`, `src/`, `docs/` | Preparado; pendiente publicación Git |
| Estructura de notebooks | Explicación → código → resultados → verificación | Implementado |

**Proyectado:** clases y algoritmos de F3, comunicación final de F4. No se atribuyen commits históricos ni evaluaciones del docente a esta revisión.
